# 04 — Nhiệm vụ 3: MedCLIP + LoRA few-shot

Đóng băng toàn bộ MedCLIP, chỉ train LoRA (rank thấp) chèn vào Q/K/V attention của cả vision (Swin) và text (BERT) encoder. Đây là phần kỹ thuật cốt lõi của đề tài.

Chạy **sweep nhiều mức shot × nhiều seed** (1/2/4/8/16-shot, mỗi mức 3 seed — cấu hình ở `FEWSHOT_SWEEP_SHOTS`/`FEWSHOT_SWEEP_SEEDS` trong `configs/config.py`) thay vì 1 con số duy nhất — đúng chuẩn báo cáo few-shot VLM (CoOp, CLIP-LoRA): thấy được đường cong accuracy cải thiện theo số shot kèm độ lệch chuẩn (mean±std), không phải 1 điểm dữ liệu đơn lẻ do may rủi bốc trúng ảnh dễ/khó.

Yêu cầu: đã chạy `01_prepare_split.ipynb` trước đó. Nên chạy trên GPU (Colab Runtime > Change runtime type > GPU) — sweep đủ 5 mức shot × 3 seed = 15 lần train, CPU sẽ rất lâu.


In [ ]:
# Cell cài đặt — chạy trên Google Colab.
# Bỏ qua nếu chạy local (đã cài sẵn requirements + có sys.path đúng).

# 1) Mount Google Drive (nếu dataset/checkpoint lưu trên Drive)
# from google.colab import drive
# drive.mount('/content/drive')

# 2) Clone / trỏ tới thư mục project (sửa lại đường dẫn cho đúng chỗ bạn để code)
PROJECT_ROOT = '/content/drive/MyDrive/Code'  # <-- sửa lại nếu khác

# 3) Cài MedCLIP (editable) + dependency của src/
# !pip install -e {PROJECT_ROOT}/MedCLIP --no-deps -q
# !pip install -r {PROJECT_ROOT}/src/requirements.txt -q

# 4) Thêm src/ vào sys.path để import được các module (configs, data, models, pipelines...)
import sys
sys.path.insert(0, f'{PROJECT_ROOT}/src')


In [ ]:
from pipelines.lora_train import run_lora_shots_sweep

# Mặc định chạy FEWSHOT_SWEEP_SHOTS x FEWSHOT_SWEEP_SEEDS trong configs/config.py
# (1/2/4/8/16-shot x 3 seed = 15 lần train). Muốn chạy nhanh hơn để thử trước, giảm bớt, ví dụ:
# results = run_lora_shots_sweep(shots_list=[4, 8, 16], seeds=[1], r=8, target='vision')
results = run_lora_shots_sweep()
results


### Confusion matrix (mức shot lớn nhất, seed đầu tiên trong sweep)


In [ ]:
from IPython.display import Image
from configs import config as cfg
import os

# Mỗi seed lưu confusion matrix riêng (lora_fewshot_{n}shot_seed{s}_confusion_matrix.png);
# file gộp lora_fewshot_{n}shot_metrics.json chỉ gộp SỐ (mean+-std), không gộp ảnh.
headline_shots = max(cfg.FEWSHOT_SWEEP_SHOTS)
first_seed = cfg.FEWSHOT_SWEEP_SEEDS[0]
Image(os.path.join(cfg.RESULTS_DIR, f'lora_fewshot_{headline_shots}shot_seed{first_seed}_confusion_matrix.png'))
